In [1]:
import os
import shutil

print("Searching for RDKit wheel...")
rdkit_files = []
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        # find any file containing 'rdkit'
        if 'rdkit' in f.lower():
            rdkit_files.append(os.path.join(root, f))

if not rdkit_files:
    print("RDKit wheel not found. Check your dataset upload.")
else:
    target_file = rdkit_files[0]
    print(f"Found: {target_file}")
    
    if not target_file.endswith('.whl'):
        safe_path = '/kaggle/working/rdkit_fixed.whl'
        shutil.copy(target_file, safe_path)
        target_file = safe_path
        print(f"Renamed to valid .whl path: {target_file}")
        
    print("Installing...")
    !pip install --no-index --no-deps {target_file}


Searching for RDKit wheel...
Found: /kaggle/input/datasets/jooseok/my-rdkit-cp312/rdkit-2026.3.1-cp312-cp312-manylinux_2_28_x86_64.whl
Installing...
Processing /kaggle/input/datasets/jooseok/my-rdkit-cp312/rdkit-2026.3.1-cp312-cp312-manylinux_2_28_x86_64.whl


In [2]:
import os
import glob
import pandas as pd
import numpy as np
import lightgbm as lgb
from rdkit import Chem
from rdkit.Chem import Descriptors
from sklearn.model_selection import KFold
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')


def get_rdkit_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return [np.nan] * len(Descriptors._descList)
    
    descriptors = []
    for _, desc_func in Descriptors._descList:
        try:
            descriptors.append(desc_func(mol))
        except:
            descriptors.append(np.nan)
    return descriptors


def main():
    print("Loading data...")

    found_train = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    if not found_train:
        print("train.csv not found.")
        return
        
    data_dir = os.path.dirname(found_train[0])
    print(f"Data path: {data_dir}")
    
    train_df = pd.read_csv(f'{data_dir}/train.csv')
    test_df = pd.read_csv(f'{data_dir}/test.csv')
    
    smiles_column = 'SMILES'
    target_columns = ['Tg', 'FFV', 'Tc', 'Density', 'Rg']
    
    print("Extracting descriptors...")
    tqdm.pandas()
    
    train_descriptors = train_df[smiles_column].progress_apply(get_rdkit_descriptors)
    desc_names = [d[0] for d in Descriptors._descList]
    X_all = pd.DataFrame(train_descriptors.tolist(), columns=desc_names)
    
    test_descriptors = test_df[smiles_column].progress_apply(get_rdkit_descriptors)
    X_submit = pd.DataFrame(test_descriptors.tolist(), columns=desc_names)
    
    submission = pd.DataFrame({'id': test_df['id']})

    params = {
        'objective': 'regression_l1',
        'metric': 'l1',
        'boosting_type': 'gbdt',
        'learning_rate': 0.05,
        'num_leaves': 31,
        'random_state': 42,
        'verbose': -1
    }

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    oof_maes = {}

    print("Training models (OOF 5-Fold)...")
    for target in target_columns:
        print(f"Target: {target}")
        
        y_all = train_df[target]
        valid_idx = y_all.dropna().index
        X = X_all.loc[valid_idx].values
        y = y_all.loc[valid_idx].values
        
        if len(y) == 0:
            submission[target] = 0
            continue
        
        oof_preds = np.zeros(len(y))
        test_preds_folds = []

        for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):
            train_data = lgb.Dataset(X[tr_idx], label=y[tr_idx])
            val_data   = lgb.Dataset(X[val_idx], label=y[val_idx], reference=train_data)
            
            model = lgb.train(
                params, train_data,
                num_boost_round=1000,
                valid_sets=[val_data],
                callbacks=[
                    lgb.early_stopping(stopping_rounds=50, verbose=False),
                    lgb.log_evaluation(period=-1)
                ]
            )
            oof_preds[val_idx] = model.predict(X[val_idx], num_iteration=model.best_iteration)
            test_preds_folds.append(model.predict(X_submit.values, num_iteration=model.best_iteration))
            print(f"  Fold {fold} MAE: {np.abs(y[val_idx] - oof_preds[val_idx]).mean():.4f}")

        oof_mae = np.abs(y - oof_preds).mean()
        oof_maes[target] = oof_mae
        print(f"  => OOF MAE: {oof_mae:.4f}")
        submission[target] = np.mean(test_preds_folds, axis=0)

    # OOF wMAE
    PROPERTY_WEIGHTS = {'Tg': 1.0, 'FFV': 10.0, 'Tc': 1.0, 'Density': 1.0, 'Rg': 1.0}
    wmae = sum(oof_maes[t] * PROPERTY_WEIGHTS[t] for t in oof_maes) / sum(PROPERTY_WEIGHTS[t] for t in oof_maes)
    print(f"\nOOF wMAE: {wmae:.4f}")

    print("Saving submission...")
    submission = submission[['id'] + target_columns]
    submission.to_csv('submission.csv', index=False)
    print("Done.")


if __name__ == "__main__":
    main()


Loading data...
Data path: /kaggle/input/competitions/neurips-open-polymer-prediction-2025
Extracting descriptors...


100%|██████████| 3/3 [00:00<00:00, 53.78it/s]


Training models (OOF 5-Fold)...
Target: Tg
  Fold 1 MAE: 50.9205
  Fold 2 MAE: 59.8021
  Fold 3 MAE: 54.7918
  Fold 4 MAE: 42.6742
  Fold 5 MAE: 49.9281
  => OOF MAE: 51.6220
Target: FFV
  Fold 1 MAE: 0.0071
  Fold 2 MAE: 0.0069
  Fold 3 MAE: 0.0068
  Fold 4 MAE: 0.0064
  Fold 5 MAE: 0.0070
  => OOF MAE: 0.0068
Target: Tc
  Fold 1 MAE: 0.0275
  Fold 2 MAE: 0.0268
  Fold 3 MAE: 0.0292
  Fold 4 MAE: 0.0231
  Fold 5 MAE: 0.0274
  => OOF MAE: 0.0268
Target: Density
  Fold 1 MAE: 0.0323
  Fold 2 MAE: 0.0504
  Fold 3 MAE: 0.0311
  Fold 4 MAE: 0.0238
  Fold 5 MAE: 0.0330
  => OOF MAE: 0.0341
Target: Rg
  Fold 1 MAE: 2.0036
  Fold 2 MAE: 1.8854
  Fold 3 MAE: 2.0235
  Fold 4 MAE: 1.5612
  Fold 5 MAE: 1.7431
  => OOF MAE: 1.8435

OOF wMAE: 3.8282
Saving submission...
Done.
